# Resources

In [ ]:
!gdown 1-1MvuX_vjv5z6ciO3TolASXQ83iSpS-m
!gdown 1r0RvuRlVTFMupEmRi2QJOaErucui-N14
!gdown 1mXbNV1J5rRUGx0uEIe21qy0rREcmJJczctvzfgdt1Oc
!gdown 1--rLV7qE_T1Le5XtG4Ug8G7Cl1InhX7m
!gdown 1-2Z0bodWxlLNSjt-CPw529-0jKplizIE
!gdown 1-1MvuX_vjv5z6ciO3TolASXQ83iSpS-m

Downloading...
From: https://drive.google.com/uc?id=1-1MvuX_vjv5z6ciO3TolASXQ83iSpS-m
To: /content/data_shop.csv
100% 73.8k/73.8k [00:00<00:00, 97.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1r0RvuRlVTFMupEmRi2QJOaErucui-N14
To: /content/data_reviews_purchase.csv
100% 98.7M/98.7M [00:00<00:00, 168MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1mXbNV1J5rRUGx0uEIe21qy0rREcmJJczctvzfgdt1Oc
From (redirected): https://docs.google.com/spreadsheets/d/1mXbNV1J5rRUGx0uEIe21qy0rREcmJJczctvzfgdt1Oc/export?format=xlsx
To: /content/data_reviews_purchase.xlsx
30.8MB [00:00, 62.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1--rLV7qE_T1Le5XtG4Ug8G7Cl1InhX7m
To: /content/data_product.csv
100% 4.18M/4.18M [00:00<00:00, 171MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-2Z0bodWxlLNSjt-CPw529-0jKplizIE
To: /content/data_product_attribute.csv
100% 964k/964k [00:00<00:00, 114MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-1MvuX_vjv5z6

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

In [ ]:
# Read in data
ratings=pd.read_csv('/content/data_reviews_purchase.csv')

# Take a look at the data
ratings.head()

,Unnamed: 0,user_id,product_id,rating,product_name_x,cmt_date,shop_id,variation_x,product_quality,processed_comment
0,0,0,0,4,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2023-01-06 12:28:25,0,Simple,4.0,mùi hương ko bít dành cho da tùy da công da an...
1,1,1,0,5,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2023-01-03 12:18:12,0,Simple,5.0,mùi nhẹ dành cho da mọi loại da công_dụng rửa ...
2,2,2,0,5,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2023-01-05 04:57:08,0,Simple,5.0,công_dụng rửa mặt_hàng giao rất nhanh sản_phẩm...
3,3,3,0,5,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2023-01-06 12:07:48,0,Simple,5.0,lần thứ 2 mua sp simple thì ai cũng biết lành_...
4,4,4,0,5,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2022-12-31 06:21:50,0,Simple,5.0,mua lần hai rồi dùng rất thích mọi người nên m...


There are four columns in the ratings dataset, userID, movieID, rating, and timestamp.

The dataset has over 100k records, and there is no missing data.

## Userbased

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm  # Thư viện để hiển thị thanh tiến trình

###### Step 1: Tải và Chuẩn Bị Dữ Liệu


def load_and_prepare_data(filepath, min_ratings_threshold=5):
    """Tải dữ liệu, làm sạch và lọc người dùng có ít đánh giá."""
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(
            f"LỖI: Không tìm thấy file '{filepath}'. Vui lòng kiểm tra lại đường dẫn."
        )
        return None

    # Làm sạch cơ bản
    if "Unnamed: 0" in df.columns:
        df = df.drop("Unnamed: 0", axis=1)
    if "product_name_x" in df.columns:
        df.rename(columns={"product_name_x": "product_id"}, inplace=True)

    # Print columns để debug
    print(f"Columns trong dataset: {df.columns.tolist()}")

    # Kiểm tra bắt buộc có các cột cần thiết
    required_columns = ["user_id", "product_id", "rating"]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        print(f"ERROR: Missing required columns: {missing_columns}")
        return None

    print(f"Số lượng đánh giá ban đầu: {len(df)}")
    print(f"Số lượng người dùng ban đầu: {df['user_id'].nunique()}")
    print(f"Số lượng sản phẩm: {df['product_id'].nunique()}")

    # Lọc người dùng có ít hơn 'min_ratings_threshold' đánh giá
    user_counts = df["user_id"].value_counts()
    print(
        f"\nNgưỡng số đánh giá tối thiểu để giữ lại người dùng: {int(min_ratings_threshold)}"
    )

    active_users = user_counts[user_counts >= min_ratings_threshold].index
    df_filtered = df[df["user_id"].isin(active_users)]

    print(f"Số lượng đánh giá sau khi lọc người dùng: {len(df_filtered)}")
    print(f"Số lượng người dùng sau khi lọc: {df_filtered['user_id'].nunique()}")

    return df_filtered


from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm  # Thư viện để hiển thị thanh tiến trình

###### Step 1: Tải và Chuẩn Bị Dữ Liệu


def load_and_prepare_data(filepath, min_ratings_threshold=5):
    """Tải dữ liệu, làm sạch và lọc người dùng có ít đánh giá."""
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(
            f"LỖI: Không tìm thấy file '{filepath}'. Vui lòng kiểm tra lại đường dẫn."
        )
        return None

    # Làm sạch cơ bản
    if "Unnamed: 0" in df.columns:
        df = df.drop("Unnamed: 0", axis=1)
    if "product_name_x" in df.columns:
        df.rename(columns={"product_name_x": "product_name"}, inplace=True)

    print(f"Số lượng đánh giá ban đầu: {len(df)}")
    print(f"Số lượng người dùng ban đầu: {df['user_id'].nunique()}")
    print(f"Số lượng sản phẩm: {df['product_id'].nunique()}")

    # Lọc người dùng có ít hơn 'min_ratings_threshold' đánh giá
    user_counts = df["user_id"].value_counts()
    print(
        f"\nNgưỡng số đánh giá tối thiểu để giữ lại người dùng: {int(min_ratings_threshold)}"
    )

    active_users = user_counts[user_counts >= min_ratings_threshold].index
    df_filtered = df[df["user_id"].isin(active_users)]

    print(f"Số lượng đánh giá sau khi lọc người dùng: {len(df_filtered)}")
    print(f"Số lượng người dùng sau khi lọc: {df_filtered['user_id'].nunique()}")

    return df_filtered


###### Step 2: Chia Dữ Liệu thành Tập Train và Test


def create_train_test_split(df, test_size=0.2):
    """
    Chia dữ liệu theo từng người dùng để đảm bảo mỗi người dùng
    đều có dữ liệu trong cả tập train và test.
    """
    train_list = []
    test_list = []

    for user_id, group in tqdm(df.groupby("user_id"), desc="Đang chia Train/Test..."):
        test_part = group.sample(frac=test_size, random_state=42)
        train_part = group.drop(test_part.index)
        train_list.append(train_part)
        test_list.append(test_part)

    train_df = pd.concat(train_list)
    test_df = pd.concat(test_list)

    print("\n--- Kích Thước Dữ Liệu ---")
    print(f"Tập Train: {len(train_df)} đánh giá")
    print(f"Tập Test: {len(test_df)} đánh giá")

    return train_df, test_df


###### Step 3: Xây Dựng và "Huấn Luyện" Mô Hình


class CollaborativeFilteringModel:
    def __init__(self):
        self.user_similarity = None
        self.matrix_norm = None
        self.train_matrix = None
        self.user_mean_ratings = None
        self.prediction_matrix = (
            None  # New attribute to store the full prediction matrix
        )

    def fit(self, train_df):
        """Xây dựng ma trận tương đồng từ tập train."""
        print("\nĐang huấn luyện mô hình (xây dựng ma trận)...")
        self.train_matrix = train_df.pivot_table(
            index="user_id", columns="product_id", values="rating"
        )
        self.user_mean_ratings = self.train_matrix.mean(axis=1)
        self.matrix_norm = self.train_matrix.subtract(
            self.user_mean_ratings, axis="rows"
        )
        self.user_similarity = self.matrix_norm.T.corr().fillna(0)

    def predict(self, user_id, product_id, n_similar_users=15):
        """Dự đoán rating của một user cho một sản phẩm."""
        if (
            user_id not in self.user_similarity
            or product_id not in self.matrix_norm.columns
        ):
            return self.user_mean_ratings.get(user_id, np.nan)

        product_ratings = self.matrix_norm[product_id]
        similarities = self.user_similarity[user_id]
        rated_users = product_ratings.dropna().index

        valid_similarities = similarities.loc[
            similarities.index.intersection(rated_users)
        ]
        valid_ratings = product_ratings.loc[
            product_ratings.index.intersection(rated_users)
        ]

        if valid_similarities.empty:
            return self.user_mean_ratings.get(user_id, np.nan)

        top_users = valid_similarities.nlargest(n_similar_users)

        numerator = (top_users * valid_ratings.loc[top_users.index]).sum()
        denominator = top_users.sum()

        if denominator == 0:
            return self.user_mean_ratings.get(user_id, np.nan)

        predicted_score = self.user_mean_ratings[user_id] + (numerator / denominator)
        return predicted_score

    def recommend_top_k(self, user_id, k=10):
        """Gợi ý top-k sản phẩm cho một user."""
        try:
            rated_products = self.train_matrix.loc[user_id].dropna().index
        except KeyError:
            return []  # User không có trong tập train

        all_products = self.train_matrix.columns
        products_to_recommend = all_products.drop(rated_products, errors="ignore")

        predictions = {}
        for product in products_to_recommend:
            pred = self.predict(user_id, product)
            if not np.isnan(pred):
                predictions[product] = pred

        sorted_predictions = sorted(
            predictions.items(), key=lambda item: item[1], reverse=True
        )
        return [product for product, score in sorted_predictions[:k]]

    def predict_full_matrix(self, n_similar_users=15):
        """
        Dự đoán rating cho tất cả các cặp user-product chưa có trong ma trận train.
        Kết quả được lưu vào thuộc tính prediction_matrix.
        """
        print("\nĐang dự đoán toàn bộ ma trận user-product...")

        # Tạo ma trận dự đoán với kích thước giống train_matrix
        self.prediction_matrix = self.train_matrix.copy()

        # Duyệt qua từng user
        for user_id in tqdm(
            self.train_matrix.index, desc="Đang dự đoán cho các user..."
        ):
            # Lấy các sản phẩm mà user chưa đánh giá
            rated_products = self.train_matrix.loc[user_id].dropna().index
            unrated_products = self.train_matrix.columns.difference(rated_products)

            # Dự đoán cho tất cả các sản phẩm chưa đánh giá
            for product_id in unrated_products:
                pred = self.predict(user_id, product_id, n_similar_users)
                self.prediction_matrix.loc[user_id, product_id] = pred

        print("Đã hoàn thành dự đoán toàn bộ ma trận!")
        return self.prediction_matrix

    def recommend_from_matrix(self, user_id, k=10):
        """
        Gợi ý top-k sản phẩm cho một user dựa trên ma trận dự đoán đã tính toán trước.
        """
        # Kiểm tra xem đã tính toán ma trận dự đoán chưa
        if self.prediction_matrix is None:
            print("Ma trận dự đoán chưa được tính toán. Đang tính toán...")
            self.predict_full_matrix()

        try:
            # Lấy các sản phẩm mà user đã đánh giá
            rated_products = self.train_matrix.loc[user_id].dropna().index

            # Lấy dự đoán cho tất cả sản phẩm user chưa đánh giá
            user_predictions = self.prediction_matrix.loc[user_id].drop(
                rated_products, errors="ignore"
            )

            # Lọc bỏ các giá trị NaN
            user_predictions = user_predictions.dropna()

            # Nếu không có dự đoán nào, trả về danh sách rỗng
            if len(user_predictions) == 0:
                return []

            # Sắp xếp theo điểm dự đoán giảm dần và lấy top-k
            top_products = user_predictions.nlargest(k).index.tolist()
            return top_products
        except KeyError:
            # User không có trong tập train
            return []


###### Step 4: Đánh Giá Mô Hình (Bổ sung F1 và Coverage)


def evaluate_model(model, train_df, test_df, k=10):
    """Tính toán tất cả các chỉ số: MAE, RMSE, Precision, Recall, F1, Coverage."""
    print("\nBắt đầu đánh giá mô hình trên tập Test...")

    # --- Đánh giá MAE và RMSE ---
    predictions = []
    actuals = []
    total_predictions = 0
    invalid_predictions = 0

    # Debugging: Print some info about test data
    print(f"Số lượng dòng trong tập test: {len(test_df)}")
    print(f"Columns trong test_df: {test_df.columns.tolist()}")

    for row in tqdm(test_df.itertuples(), desc="Tính MAE/RMSE...", total=len(test_df)):
        total_predictions += 1
        try:
            # Check if the required attributes exist
            if not hasattr(row, "user_id") or not hasattr(row, "product_id"):
                print(f"Missing required columns in row: {row}")
                continue

            pred = model.predict(row.user_id, row.product_id)
            predictions.append(pred)
            actuals.append(row.rating)
        except Exception as e:
            print(f"Error processing row: {e}")
            if invalid_predictions <= 5:
                print(f"Problematic row: {row}")
            invalid_predictions += 1

    print(f"Total predictions attempted: {total_predictions}")
    print(f"Valid predictions: {len(predictions)}")
    print(f"Invalid predictions: {invalid_predictions}")

    mae = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))

    # In kết quả
    print("\n==============================================")
    print("      KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH HOÀN CHỈNH      ")
    print("==============================================")
    print(f"MAE (Mean Absolute Error):      {mae:.4f}")
    print(f"RMSE (Root Mean Squared Error):   {rmse:.4f}")


###### Step 5: Lưu và Tải Ma Trận Dự Đoán để Sử Dụng trong Hệ Thống Gợi Ý


def save_prediction_matrix(model, filepath):
    """
    Lưu ma trận dự đoán và các thông tin cần thiết để sử dụng lại sau này.
    """
    if model.prediction_matrix is None:
        print("Ma trận dự đoán chưa được tính toán. Vui lòng tính toán trước khi lưu.")
        return False

    # Tạo dictionary chứa các thông tin cần thiết
    data_to_save = {
        "prediction_matrix": model.prediction_matrix.to_dict(),
        "train_matrix": model.train_matrix.to_dict(),
        "timestamp": time.time(),
    }

    # Lưu vào file
    try:
        with open(filepath, "wb") as f:
            np.save(f, data_to_save)
        print(f"Đã lưu ma trận dự đoán thành công vào {filepath}")
        return True
    except Exception as e:
        print(f"Lỗi khi lưu ma trận dự đoán: {e}")
        return False


def load_prediction_data(filepath):
    """
    Tải ma trận dự đoán và các thông tin liên quan từ file đã lưu.
    """
    try:
        with open(filepath, "rb") as f:
            loaded_data = np.load(f, allow_pickle=True).item()

        # Chuyển đổi từ dict sang DataFrame
        prediction_matrix = pd.DataFrame.from_dict(loaded_data["prediction_matrix"])
        train_matrix = pd.DataFrame.from_dict(loaded_data["train_matrix"])
        timestamp = loaded_data["timestamp"]

        print(f"Đã tải ma trận dự đoán từ {filepath}")
        print(f"Thời gian tạo: {time.ctime(timestamp)}")

        return prediction_matrix, train_matrix
    except Exception as e:
        print(f"Lỗi khi tải ma trận dự đoán: {e}")
        return None, None


def recommend_for_user(user_id, prediction_matrix, train_matrix, k=10):
    """
    Hàm gợi ý độc lập, không cần khởi tạo và huấn luyện mô hình.
    Chỉ cần cung cấp ma trận dự đoán đã được tính toán trước.

    Args:
        user_id: ID của người dùng cần gợi ý
        prediction_matrix: Ma trận dự đoán đã được tính toán và lưu trước đó
        train_matrix: Ma trận huấn luyện để biết sản phẩm nào người dùng đã đánh giá
        k: Số lượng sản phẩm muốn gợi ý

    Returns:
        Danh sách top-k sản phẩm được gợi ý
    """
    try:
        # Kiểm tra xem user có trong ma trận không
        if user_id not in prediction_matrix.index:
            print(f"User ID {user_id} không tồn tại trong dữ liệu.")
            return []

        # Lấy các sản phẩm mà user đã đánh giá
        rated_products = train_matrix.loc[user_id].dropna().index

        # Lấy dự đoán cho tất cả sản phẩm user chưa đánh giá
        user_predictions = prediction_matrix.loc[user_id].drop(
            rated_products, errors="ignore"
        )

        # Lọc bỏ các giá trị NaN
        user_predictions = user_predictions.dropna()

        # Nếu không có dự đoán nào, trả về danh sách rỗng
        if len(user_predictions) == 0:
            return []

        # Sắp xếp theo điểm dự đoán giảm dần và lấy top-k
        top_products = user_predictions.nlargest(k).index.tolist()
        return top_products
    except Exception as e:
        print(f"Lỗi khi gợi ý cho user {user_id}: {e}")
        return []


if __name__ == "__main__":
    start_time = time.time()

    # Đường dẫn đến file dữ liệu đánh giá của bạn
    filepath = "data_reviews_purchase.csv"

    # Đường dẫn để lưu/tải ma trận dự đoán
    matrix_save_path = "prediction_matrix.npy"

    # Cờ để quyết định có huấn luyện lại mô hình hay tải từ file đã lưu
    retrain_model = True  # Đặt thành False nếu muốn chỉ tải ma trận đã lưu

    if retrain_model:
        print("\n===== HUẤN LUYỆN MÔ HÌNH VÀ TÍNH TOÁN MA TRẬN DỰ ĐOÁN =====")
        # Bước 1
        df_filtered = load_and_prepare_data(filepath, min_ratings_threshold=4)

        if df_filtered is not None:
            # Bước 2
            train_set, test_set = create_train_test_split(df_filtered, test_size=0.2)

            # Bước 3
            model = CollaborativeFilteringModel()
            model.fit(train_set)

            # Kiểm tra một số thông tin về ma trận đã xây dựng
            print(f"\nThông tin về ma trận đã xây dựng:")
            print(f"Shape của train_matrix: {model.train_matrix.shape}")

            # Dự đoán toàn bộ ma trận user-product
            model.predict_full_matrix(n_similar_users=15)

            # Thử nghiệm chức năng gợi ý với một vài user
            print("\nThử nghiệm chức năng gợi ý trong mô hình:")
            sample_users = (
                train_set["user_id"]
                .sample(min(3, train_set["user_id"].nunique()))
                .unique()
            )
            for user_id in sample_users:
                print(f"\nGợi ý cho user_id: {user_id}")
                recommendations = model.recommend_from_matrix(user_id, k=5)
                print(f"Top 5 sản phẩm gợi ý: {recommendations}")

            # Bước 4
            evaluate_model(model, train_set, test_set, k=10)

            # Bước 5: Lưu ma trận dự đoán để sử dụng sau này
            save_prediction_matrix(model, matrix_save_path)




===== HUẤN LUYỆN MÔ HÌNH VÀ TÍNH TOÁN MA TRẬN DỰ ĐOÁN =====
Số lượng đánh giá ban đầu: 369099
Số lượng người dùng ban đầu: 304708
Số lượng sản phẩm: 2231

Ngưỡng số đánh giá tối thiểu để giữ lại người dùng: 4
Số lượng đánh giá sau khi lọc người dùng: 17494
Số lượng người dùng sau khi lọc: 3688


Đang chia Train/Test...: 100%|██████████| 3688/3688 [00:02<00:00, 1313.24it/s]



--- Kích Thước Dữ Liệu ---
Tập Train: 13615 đánh giá
Tập Test: 3879 đánh giá

Đang huấn luyện mô hình (xây dựng ma trận)...

Thông tin về ma trận đã xây dựng:
Shape của train_matrix: (3688, 1127)

Đang dự đoán toàn bộ ma trận user-product...


Đang dự đoán cho các user...: 100%|██████████| 3688/3688 [1:29:42<00:00,  1.46s/it]


Đã hoàn thành dự đoán toàn bộ ma trận!

Thử nghiệm chức năng gợi ý trong mô hình:

Gợi ý cho user_id: 107622
Top 5 sản phẩm gợi ý: [0, 1, 2, 3, 4]

Gợi ý cho user_id: 186408
Top 5 sản phẩm gợi ý: [0, 1, 2, 3, 4]

Gợi ý cho user_id: 96899
Top 5 sản phẩm gợi ý: [0, 1, 2, 3, 4]

Bắt đầu đánh giá mô hình trên tập Test...
Số lượng dòng trong tập test: 3879
Columns trong test_df: ['user_id', 'product_id', 'rating', 'product_name', 'cmt_date', 'shop_id', 'variation_x', 'product_quality', 'processed_comment']


Tính MAE/RMSE...: 100%|██████████| 3879/3879 [00:05<00:00, 731.16it/s]


Total predictions attempted: 3879
Valid predictions: 3879
Invalid predictions: 0

      KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH HOÀN CHỈNH      
MAE (Mean Absolute Error):      0.0550
RMSE (Root Mean Squared Error):   0.3030
Đã lưu ma trận dự đoán thành công vào prediction_matrix.npy


In [ ]:
    # ===== DEMO TÍCH HỢP VÀO HỆ THỐNG GỢI Ý =====
    print("\n\n===== DEMO TÍCH HỢP VÀO HỆ THỐNG GỢI Ý =====")
    print("Bước 1: Tải ma trận dự đoán đã lưu")
    prediction_matrix, train_matrix = load_prediction_data(matrix_save_path)

    if prediction_matrix is not None and train_matrix is not None:
        print("\nBước 2: Sử dụng hàm gợi ý độc lập để tạo gợi ý cho người dùng")

        # Lấy một số user ngẫu nhiên để thử nghiệm
        sample_user_ids = np.random.choice(
            prediction_matrix.index,
            size=min(3, len(prediction_matrix.index)),
            replace=False,
        )

        print(f"Ma trận dự đoán có shape: {prediction_matrix.shape}")
        print(f"Số lượng người dùng: {len(prediction_matrix.index)}")
        print(f"Số lượng sản phẩm: {len(prediction_matrix.columns)}")

        # Thử nghiệm hàm gợi ý độc lập (không cần mô hình)
        print("\nTạo gợi ý cho các user mẫu:")
        for i, user_id in enumerate(sample_user_ids, 1):
            print(f"\n{i}. Gợi ý cho user_id: {user_id}")
            recommendations = recommend_for_user(
                user_id, prediction_matrix, train_matrix, k=5
            )
            print(f"   Top 5 sản phẩm gợi ý: {recommendations}")

        # Giả lập tích hợp vào API hoặc ứng dụng web
        print("\n===== MÔ PHỎNG TÍCH HỢP VÀO API =====")
        print("Ví dụ mã API endpoint để trả về gợi ý sản phẩm:")
        print(
            """
def get_recommendations_api(user_id, k=10):
    # Đã tải ma trận dự đoán khi khởi động server
    # prediction_matrix, train_matrix = load_prediction_data("data/prediction_matrix.npy")

    # Gọi hàm gợi ý độc lập
    recommendations = recommend_for_user(user_id, prediction_matrix, train_matrix, k)

    # Trả về danh sách sản phẩm được gợi ý
    return recommendations
        """
        )

        # Mô phỏng gọi API
        print("\nGọi API gợi ý cho user_id:", sample_user_ids[0])
        recommendations = recommend_for_user(
            sample_user_ids[0], prediction_matrix, train_matrix, k=10
        )
        print("Response từ API:")
        print(recommendations)

    end_time = time.time()
    print(f"\nTổng thời gian thực thi: {end_time - start_time:.2f} giây")



===== DEMO TÍCH HỢP VÀO HỆ THỐNG GỢI Ý =====
Bước 1: Tải ma trận dự đoán đã lưu
Đã tải ma trận dự đoán từ prediction_matrix.npy
Thời gian tạo: Sat Jun 28 19:02:11 2025

Bước 2: Sử dụng hàm gợi ý độc lập để tạo gợi ý cho người dùng
Ma trận dự đoán có shape: (3688, 1127)
Số lượng người dùng: 3688
Số lượng sản phẩm: 1127

Tạo gợi ý cho các user mẫu:

1. Gợi ý cho user_id: 219428
   Top 5 sản phẩm gợi ý: [0, 1, 2, 3, 4]

2. Gợi ý cho user_id: 30027
   Top 5 sản phẩm gợi ý: [0, 1, 2, 3, 4]

3. Gợi ý cho user_id: 267
   Top 5 sản phẩm gợi ý: [1, 2, 3, 4, 10]

===== MÔ PHỎNG TÍCH HỢP VÀO API =====
Ví dụ mã API endpoint để trả về gợi ý sản phẩm:

def get_recommendations_api(user_id, k=10):
# Đã tải ma trận dự đoán khi khởi động server
# prediction_matrix, train_matrix = load_prediction_data("data/prediction_matrix.npy")

# Gọi hàm gợi ý độc lập
recommendations = recommend_for_user(user_id, prediction_matrix, train_matrix, k)

# Trả về danh sách sản phẩm được gợi ý
return recommendations
    

In [ ]:
prediction_matrix

,0,1,2,3,4,10,11,13,14,18,...,2216,2217,2218,2220,2221,2222,2227,2229,2232,2235
56,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
243,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
267,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
402,5.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
474,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303159,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
303354,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
303584,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
304098,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0


In [ ]:
# prompt: đưa ra cho tôi ma trận simalirity matrix
# ===== HUẤN LUYỆN MÔ HÌNH VÀ TÍNH TOÁN MA TRẬN DỰ ĐOÁN =====
# Số lượng đánh giá ban đầu: 369099
# Số lượng người dùng ban đầu: 304708
# Số lượng sản phẩm: 2231
# Ngưỡng số đánh giá tối thiểu để giữ lại người dùng: 4
# Số lượng đánh giá sau khi lọc người dùng: 17494
# Số lượng người dùng sau khi lọc: 3688
# Đang chia Train/Test...: 100%|██████████| 3688/3688 [00:02<00:00, 1313.24it/s]
# --- Kích Thước Dữ Liệu ---
# Tập Train: 13615 đánh giá
# Tập Test: 3879 đánh giá
# Đang huấn luyện mô hình (xây dựng ma trận)...
# Thông tin về ma trận đã xây dựng:
# Shape của train_matrix: (3688, 1127)
# Đang dự đoán toàn bộ ma trận user-product...
# Đang dự đoán cho các user...: 100%|██████████| 3688/3688 [1:29:42<00:00,  1.46s/it]
# Đã hoàn thành dự đoán toàn bộ ma trận!
# Thử nghiệm chức năng gợi ý trong mô hình:
# Gợi ý cho user_id: 107622
# Top 5 sản phẩm gợi ý: [0, 1, 2, 3, 4]
# Gợi ý cho user_id: 186408
# Top 5 sản phẩm gợi ý: [0, 1, 2, 3, 4]
# Gợi ý cho user_id: 96899
# Top 5 sản phẩm gợi ý: [0, 1, 2, 3, 4]
# Bắt đầu đánh giá mô hình trên tập Test...
# Số lượng dòng trong tập test: 3879
# Columns trong test_df: ['user_id', 'product_id', 'rating', 'product_name', 'cmt_date', 'shop_id', 'variation_x', 'product_quality', 'processed_comment']
# Tính MAE/RMSE...: 100%|██████████| 3879/3879 [00:05<00:00, 731.16it/s]
# Total predictions attempted: 3879
# Valid predictions: 3879
# Invalid predictions: 0

model.user_similarity


## Item based

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm # Thư viện để hiển thị thanh tiến trình

###### Step 1: Tải và Chuẩn Bị Dữ Liệu (Không đổi)

def load_and_prepare_data(filepath, min_ratings_threshold=5):
    """Tải dữ liệu, làm sạch và lọc người dùng có ít đánh giá."""
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"LỖI: Không tìm thấy file '{filepath}'. Vui lòng kiểm tra lại đường dẫn.")
        return None

    if 'Unnamed: 0' in df.columns:
        df = df.drop('Unnamed: 0', axis=1)
    if 'product_name_x' in df.columns:
        df.rename(columns={'product_name_x': 'product_name'}, inplace=True)

    print(f"Số lượng đánh giá ban đầu: {len(df)}")
    print(f"Số lượng người dùng ban đầu: {df['user_id'].nunique()}")

    user_counts = df['user_id'].value_counts()
    print(f"\nNgưỡng số đánh giá tối thiểu để giữ lại người dùng: {int(min_ratings_threshold)}")

    active_users = user_counts[user_counts >= min_ratings_threshold].index
    df_filtered = df[df['user_id'].isin(active_users)]

    print(f"Số lượng đánh giá sau khi lọc người dùng: {len(df_filtered)}")
    print(f"Số lượng người dùng sau khi lọc: {df_filtered['user_id'].nunique()}")

    return df_filtered

###### Step 2: Chia Dữ Liệu thành Tập Train và Test (Không đổi)

def create_train_test_split(df, test_size=0.2):
    """Chia dữ liệu theo từng người dùng."""
    train_list = []
    test_list = []

    for user_id, group in tqdm(df.groupby('user_id'), desc="Đang chia Train/Test..."):
        test_part = group.sample(frac=test_size, random_state=42)
        train_part = group.drop(test_part.index)
        train_list.append(train_part)
        test_list.append(test_part)

    train_df = pd.concat(train_list)
    test_df = pd.concat(test_list)

    print("\n--- Kích Thước Dữ Liệu ---")
    print(f"Tập Train: {len(train_df)} đánh giá")
    print(f"Tập Test: {len(test_df)} đánh giá")

    return train_df, test_df

###### Step 3: Xây Dựng và "Huấn Luyện" Mô Hình ITEM-BASED

class ItemBasedCollaborativeFilteringModel:
    def __init__(self):
        self.item_similarity = None
        self.train_matrix = None
        self.user_mean_ratings = None

    def fit(self, train_df):
        """Xây dựng ma trận tương đồng ITEM-ITEM từ tập train."""
        print("\nĐang huấn luyện mô hình (xây dựng ma trận ITEM-ITEM)...")
        self.train_matrix = train_df.pivot_table(index='user_id', columns='product_name', values='rating')
        self.user_mean_ratings = self.train_matrix.mean(axis=1)

        matrix_norm = self.train_matrix.subtract(self.user_mean_ratings, axis='rows')
        self.item_similarity = matrix_norm.corr(method='pearson').fillna(0)

    def predict(self, user_id, product_name, n_similar_items=15):
        """Dự đoán rating của một user cho một sản phẩm dựa trên các item tương tự."""
        # Fallback to user's average rating if user or item is new
        if user_id not in self.user_mean_ratings.index or product_name not in self.item_similarity.columns:
            return self.user_mean_ratings.get(user_id, np.nan)

        similarities = self.item_similarity[product_name]
        user_ratings = self.train_matrix.loc[user_id].dropna()

        # Find items rated by the user that are also in the similarity list
        rated_similar_items = similarities.index.intersection(user_ratings.index)

        if rated_similar_items.empty:
             # If no similar items were rated by the user, return user's average rating
             return self.user_mean_ratings.get(user_id, np.nan)


        valid_similarities = similarities.loc[rated_similar_items]
        valid_ratings = user_ratings.loc[rated_similar_items]

        # Get top N similar items that the user has rated
        # Ensure we only consider items with positive similarity for the weighted sum
        top_items_mask = valid_similarities > 0
        top_items = valid_similarities[top_items_mask].nlargest(n_similar_items)


        if top_items.empty:
            # If no positively correlated similar items were rated, return user's average rating
            return self.user_mean_ratings.get(user_id, np.nan)

        # Calculate weighted sum of ratings
        numerator = (top_items * valid_ratings.loc[top_items.index]).sum()
        denominator = top_items.sum()

        # Handle case where denominator is zero (e.g., all top similarities are negative or zero,
        # but filtered above by top_items_mask) - though the nlargest(positive similarities) should handle this.
        # Added a small epsilon to denominator just in case, though should not be needed with top_items_mask
        epsilon = 1e-9
        if denominator == 0: # Should ideally not happen with top_items based on positive similarities
             return self.user_mean_ratings.get(user_id, np.nan)


        predicted_score = numerator / denominator
        # Item-based often predicts deviations from mean, so add user's mean back
        # However, the prediction formula is usually SUM(sim * rating) / SUM(sim).
        # If using normalized ratings (rating - mean), the formula is different.
        # Let's re-check standard item-based prediction formula for ratings directly.
        # It's typically (sum(sim(i,j) * r(u,j))) / sum(|sim(i,j)|) for item i, user u, over rated items j.
        # Let's use the formula for raw ratings: sum(sim * rating) / sum(|sim|)
        # The current calculation was numerator/denominator using 'top_items' which were positive correlations.
        # The standard formula is sum(sim * rating) / sum(ABS(sim)).
        # Let's re-implement the prediction based on the standard formula for raw ratings.

        # Re-calculating numerator and denominator based on ALL rated similar items (not just top N initially)
        # Then take top N *after* getting all similarities and ratings.
        all_similarities_rated = similarities.loc[user_ratings.index]
        all_ratings_rated = user_ratings.loc[user_ratings.index]

        # Calculate weighted ratings and sum of absolute similarities for *all* rated similar items first
        weighted_ratings = all_similarities_rated * all_ratings_rated
        sum_abs_similarities = all_similarities_rated.abs().sum()

        if sum_abs_similarities == 0:
             return self.user_mean_ratings.get(user_id, np.nan)

        # Now, filter for top N based on similarity magnitude? Or predict using all rated?
        # Standard Item-based prediction is often a weighted average over ALL items the user rated,
        # weighted by similarity to the target item. Let's use that.

        # Re-calculating based on ALL rated items by the user, weighted by their similarity to the target product_name
        user_rated_items = self.train_matrix.loc[user_id].dropna()
        rated_item_names = user_rated_items.index

        # Get similarities of the target product_name with ALL items the user has rated
        similarities_with_rated_items = self.item_similarity.loc[rated_item_names, product_name]

        # Remove the target item itself if it's in the list
        if product_name in similarities_with_rated_items.index:
             similarities_with_rated_items = similarities_with_rated_items.drop(product_name)
             user_rated_items = user_rated_items.drop(product_name)


        # Ensure alignment of similarities and ratings after dropping
        common_items = similarities_with_rated_items.index.intersection(user_rated_items.index)
        similarities_with_rated_items = similarities_with_rated_items.loc[common_items]
        user_rated_items = user_rated_items.loc[common_items]


        # Calculate weighted sum of ratings: sum(sim(i,j) * r(u,j)) for target item i, user u, over rated items j
        numerator = (similarities_with_rated_items * user_rated_items).sum()
        # Calculate sum of absolute similarities: sum(|sim(i,j)|)
        denominator = similarities_with_rated_items.abs().sum()

        if denominator == 0:
             # If no similar items rated by user, return user's average rating
             return self.user_mean_ratings.get(user_id, np.nan)

        predicted_score = numerator / denominator

        # Clip predictions to rating scale
        predicted_score = max(1, min(5, predicted_score))


        return predicted_score


    def recommend_top_k(self, user_id, k=10):
        """Gợi ý top-k sản phẩm cho một user."""
        try:
            rated_products = self.train_matrix.loc[user_id].dropna().index
        except KeyError:
            return [] # User không có trong tập train

        all_products = self.train_matrix.columns
        products_to_recommend = all_products.drop(rated_products, errors='ignore')

        predictions = {}
        # Only predict for items that the model knows about
        known_products_to_recommend = products_to_recommend.intersection(self.item_similarity.columns)

        for product in known_products_to_recommend:
            pred = self.predict(user_id, product)
            if not np.isnan(pred):
                predictions[product] = pred

        sorted_predictions = sorted(predictions.items(), key=lambda item: item[1], reverse=True)
        return [product for product, score in sorted_predictions[:k]]


###### Step 4: Đánh Giá Mô Hình (Không đổi)

def evaluate_model(model, train_df, test_df, k=10):
    """Tính toán tất cả các chỉ số: MAE, RMSE, Precision, Recall, F1, Coverage."""
    print("\nBắt đầu đánh giá mô hình trên tập Test...")

    predictions, actuals = [], []
    # Iterate through test_df and predict for each user-item pair
    for index, row in tqdm(test_df.iterrows(), desc="Tính MAE/RMSE...", total=len(test_df)):
        user_id = row['user_id']
        product_name = row['product_name']
        actual_rating = row['rating']

        pred = model.predict(user_id, product_name)
        if not np.isnan(pred):
            predictions.append(pred)
            actuals.append(actual_rating)

    mae = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))

    # # --- Đánh giá các chỉ số xếp hạng ---
    # precisions, recalls, all_recommended_items = [], [], set()
    # # Use users who are in the test set AND have rated items in the train set
    # test_users = test_df['user_id'].unique()
    # train_users = model.train_matrix.index
    # relevant_test_users = [u for u in test_users if u in train_users]


    # for user_id in tqdm(relevant_test_users, desc=f"Tính Precision/Recall/Coverage@{k}..."):
    #     # Relevant items are those rated >= 4.0 in the test set
    #     relevant_items = set(test_df[(test_df['user_id'] == user_id) & (test_df['rating'] >= 4.0)]['product_name'])
    #     if not relevant_items:
    #         continue

    #     recommended_items = set(model.recommend_top_k(user_id, k=k))
    #     all_recommended_items.update(recommended_items)
    #     hits = len(recommended_items.intersection(relevant_items))

    #     if k > 0:
    #         precisions.append(hits / k)
    #     else:
    #         precisions.append(0)

    #     if len(relevant_items) > 0:
    #         recalls.append(hits / len(relevant_items))
    #     else:
    #         recalls.append(0)


    # avg_precision = np.mean(precisions) if precisions else 0
    # avg_recall = np.mean(recalls) if recalls else 0
    # avg_f1 = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall) if (avg_precision + avg_recall) > 0 else 0

    # # Coverage is calculated based on all unique items in the train matrix columns
    # total_items_in_train = len(model.train_matrix.columns)
    # coverage = len(all_recommended_items) / total_items_in_train if total_items_in_train > 0 else 0

    print("\n==============================================")
    print("      KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (ITEM-BASED)    ")
    print("==============================================")
    print(f"MAE (Mean Absolute Error):      {mae:.4f}")
    print(f"RMSE (Root Mean Squared Error):   {rmse:.4f}")
    # print("----------------------------------------------")
    # print(f"Precision@10:                     {avg_precision:.4f}")
    # print(f"Recall@10:                        {avg_recall:.4f}")
    # print(f"F1-Score@10:                      {avg_f1:.4f}")
    # print("----------------------------------------------")
    # print(f"Coverage:                         {coverage:.4f}")
    # print("==============================================")

###### MAIN SCRIPT
if __name__ == '__main__':
    start_time = time.time()
    filepath = 'data_reviews_purchase.csv'

    # Reduced threshold for demonstration, you can adjust this back to 4 if preferred
    df_filtered = load_and_prepare_data(filepath, min_ratings_threshold=4)

    if df_filtered is not None:
        train_set, test_set = create_train_test_split(df_filtered, test_size=0.2)

        # Use the corrected ITEM-BASED model
        model = ItemBasedCollaborativeFilteringModel()
        model.fit(train_set)

        evaluate_model(model, train_set, test_set, k=10)

    end_time = time.time()
    print(f"\nTổng thời gian thực thi: {end_time - start_time:.2f} giây")

Số lượng đánh giá ban đầu: 369099
Số lượng người dùng ban đầu: 304708

Ngưỡng số đánh giá tối thiểu để giữ lại người dùng: 4
Số lượng đánh giá sau khi lọc người dùng: 17494
Số lượng người dùng sau khi lọc: 3688


Đang chia Train/Test...: 100%|██████████| 3688/3688 [00:04<00:00, 904.40it/s] 



--- Kích Thước Dữ Liệu ---
Tập Train: 13615 đánh giá
Tập Test: 3879 đánh giá

Đang huấn luyện mô hình (xây dựng ma trận ITEM-ITEM)...

Bắt đầu đánh giá mô hình trên tập Test...


Tính MAE/RMSE...: 100%|██████████| 3879/3879 [00:13<00:00, 285.61it/s]


      KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (ITEM-BASED)    
MAE (Mean Absolute Error):      0.3160
RMSE (Root Mean Squared Error):   1.0683

Tổng thời gian thực thi: 25.16 giây
